# Chapter 8 — Assertions, Invariants, and Contracts

**Book alignment:** Debugging AI From First Principles, Chapter 8

**Question this notebook isolates:** The `clean → summarize` refactor dropped `refund_id`
for four weeks with no crash — 12,000 quietly wrong rows a night. Does a **consumer-entry
contract** convert that silent propagation into a loud, local, attributable stop on the
first bad row — and why *consumer entry* rather than the crash line or producer exit?

In [ ]:
REQUIRED = ("order_id", "subtotal", "refund_id")

def clean(orders, *, drop_child_refund_id=False):
    out = []
    for o in orders:
        row = {"order_id": o["order_id"], "subtotal": o["subtotal"]}
        if not (drop_child_refund_id and o.get("is_child")):     # the refactor: children lose the key
            row["refund_id"] = o.get("refund_id", "")
        out.append(row)
    return out

def summarize_unguarded(rows, *, silence=False):
    total = 0.0
    for r in rows:
        rid = r.get("refund_id", "") if silence else r["refund_id"]   # .get() = the 4-week silence
        total += r["subtotal"] - (5.0 if rid else 0.0)
    return round(total, 2)

def summarize_guarded(rows):
    bad = [r for r in rows if not all(k in r for k in REQUIRED)]
    if bad:
        raise ValueError(f"summarize contract violated: {len(bad)} rows missing keys; "
                         f"first offender keys={sorted(bad[0])}")
    return round(sum(r["subtotal"] - (5.0 if r["refund_id"] else 0.0) for r in rows), 2)

ORDERS      = [{"order_id": 1, "subtotal": 100.0, "refund_id": "RB-1"},
               {"order_id": 2, "subtotal": 200.0, "refund_id": "RB-2", "is_child": True}]

## 1. Two failure modes, neither of them useful: a misleading crash, or silent drift

In [ ]:
rows = clean(ORDERS, drop_child_refund_id=True)          # the child row has NO refund_id key

# mode 1: a loud crash - but it names the indexing line, inviting a .get() silence
try:
    summarize_unguarded(rows)
    crash = None
except KeyError as e:
    crash = f"KeyError {e} deep inside summarize"
print("unguarded          :", crash)

# mode 2: the .get() "fix" - no crash, wrong total, 12,000 rows/night for 4 weeks
good  = summarize_unguarded(clean(ORDERS))                        # child kept its refund
drift = summarize_unguarded(rows, silence=True)
print(f"with .get() silence : total {drift}  (should be {good}) - the program lied, quietly")
assert crash is not None and drift != good

## 2. Three contract strengths — use the weakest that still fails loudly

| Strength | Mechanism | Fires under `python -O`? | Use when |
|---|---|---|---|
| Assertion | `assert all("refund_id" in r for r in rows)` | no | dev invariants, not money paths |
| Explicit guard | `if bad: raise ValueError(...)` | yes | **money-path default** |
| Schema / type contract | dataclass / schema at the boundary | yes | repeat breaks |

In [ ]:
# the money-path default: an explicit guard at CONSUMER ENTRY
try:
    summarize_guarded(clean(ORDERS, drop_child_refund_id=True))
    raised = None
except ValueError as e:
    raised = str(e)

print(raised)
assert raised is not None and "contract violated" in raised
print("\nloud, local, and it names the handoff + the first offender - not a KeyError three frames deep")

## 3. Placement: why consumer entry beats producer exit

In [ ]:
# a producer-exit check on clean() rots the moment a SECOND producer appears:
def clean_v2(orders):                      # a new upstream path nobody added the exit-check to
    return [{"order_id": o["order_id"], "subtotal": o["subtotal"]} for o in orders]   # no refund_id at all

# consumer-entry guard still catches it, because that is where every producer's output arrives:
try:
    summarize_guarded(clean_v2(ORDERS))
    caught = False
except ValueError:
    caught = True
assert caught
print("consumer-entry guard catches the unknown future producer; a producer-exit guard would have missed clean_v2")

## 4. A postcondition for Chapter 7's pager

In [ ]:
def paginate(items, size):
    return [items[i:i + size] for i in range(0, len(items), size)]

def paginate_checked(items, size):
    pages = paginate(items, size)
    assert all(len(p) <= size for p in pages), "page over size"
    assert sum(len(p) for p in pages) == len(items), "items lost or duplicated"
    return pages

assert paginate_checked(list(range(23)), 10) == [list(range(10)), list(range(10, 20)), list(range(20, 23))]
assert paginate_checked([], 10) == []
print("a boundary rule that cannot state its postcondition is a rule nobody understood well enough to keep")

## What we earned

The four-week silent drift needed one line: a guard at `summarize`'s **entry** that scans
the schema and raises with the producer named. Consumer entry — not the crash line, not
producer exit — because that is where the *next* unknown producer's output will arrive
(`clean_v2` proves it). Every handoff Chapters 5–7 convicted implies a contract, and the
contract belongs in the code, not the ticket.

**Notebook 09 / Chapter 9** handles the failure that survives clean code and clean data:
the environment.